# Identifying stage-specific miRNA signatures via Traditional Methods

### Traditional feature-selection methods
- **Sequential Feature Selection (SFS)**
- **Variance Threshold (VT)**
- **Chi-Squared (χ²) filter**
- **LASSO (L1-regularized Logistic Regression)**

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, chi2, SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

In [2]:
label_col = "Label"
stage_codes = [1, 2, 3, 4]
stage_map = {1: "Stage_I", 2: "Stage_II", 3: "Stage_III", 4: "Stage_IV"}
k_values = [5, 10, 15]
out_dir = Path("TeamsExports/Traditional")

for method in ["SFS", "VT", "Chi2", "LASSO"]:
    (out_dir / method).mkdir(parents=True, exist_ok=True)

In [3]:
def select_sfs(X_train, y_train, feature_names, class_weight):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)
    model = LogisticRegression(max_iter=1000, random_state=42, solver='liblinear', class_weight=class_weight)
    
    sfs = SequentialFeatureSelector(model, n_features_to_select=15, 
                                      direction='forward', scoring='accuracy', cv=None)
    sfs.fit(X_scaled, y_train)
    
    selected_mask = sfs.get_support()
    selected_features = feature_names[selected_mask]
    remaining_features = feature_names[~selected_mask]
    
    remaining_X = X_scaled[:, ~selected_mask]
    scores = f_classif(remaining_X, y_train)[0]
    scores = np.nan_to_num(scores, nan=0.0)
    remaining_ranked = remaining_features[np.argsort(scores)[::-1]]
    
    return np.concatenate([selected_features, remaining_ranked])

def select_vt(X_train, y_train, feature_names, class_weight):
    selector = SelectKBest(score_func=f_classif, k='all')
    selector.fit(X_train, y_train)
    scores = selector.scores_
    scores = np.nan_to_num(scores, nan=0.0)
    ranked_indices = np.argsort(scores)[::-1]
    return feature_names[ranked_indices]

def select_chi2(X_train, y_train, feature_names, class_weight):
    X_train_pos = X_train - X_train.min(axis=0) + 1e-10
    selector = SelectKBest(score_func=chi2, k='all')
    selector.fit(X_train_pos, y_train)
    scores = selector.scores_
    scores = np.nan_to_num(scores, nan=0.0)
    ranked_indices = np.argsort(scores)[::-1]
    return feature_names[ranked_indices]

def select_lasso(X_train, y_train, feature_names, class_weight):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)
    model = LogisticRegression(penalty='l1', solver='liblinear', C=0.1, 
                                class_weight=class_weight, random_state=42, max_iter=1000)
    model.fit(X_scaled, y_train)
    coef = np.abs(model.coef_[0])
    ranked_indices = np.argsort(coef)[::-1]
    return feature_names[ranked_indices]

In [4]:
df = pd.read_csv("breast_cancer.csv")

methods = {
    "SFS": select_sfs,
    "VT": select_vt,
    "Chi2": select_chi2,
    "LASSO": select_lasso
}

for stage_code in stage_codes:
    stage_slug = stage_map[stage_code]
    
    df_stage = df[df[label_col].isin([0, stage_code])].copy()
    X = df_stage.drop(columns=[label_col])
    y = df_stage[label_col].replace({0: 0, stage_code: 1})
    feature_names = X.columns.values
    
    counts = y.value_counts().to_dict()
    minority = min(counts.values())
    test_size = 0.20 if minority < 30 else 0.30
    can_stratify = all(v >= 2 for v in counts.values())
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y if can_stratify else None, random_state=42
    )
    
    classes = np.array([0, 1])
    cw = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weight = {0: float(cw[0]), 1: float(cw[1])}
    
    for method_name, method_func in methods.items():
        ranked_features = method_func(X_train.values, y_train.values, feature_names, class_weight)
        
        for k in k_values:
            top_k = ranked_features[:k].tolist()
            pd.Series(top_k, name="miRNA").to_csv(
                out_dir / method_name / f"{method_name}_top{k}_{stage_slug}.csv",
                index=False
            )

print("Traditional feature selection complete. Files saved to TeamsExports/Traditional/")

C:\Python313\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [  58   59   60   70   74   75   85   87   92   93  100  102  103  104
  105  111  120  136  140  141  142  143  145  146  147  148  149  150
  159  162  183  186  187  204  218  250  267  268  269  304  323  324
  330  335  336  337  338  353  354  355  356  358  362  374  388  414
  425  426  427  428  430  438  457  515  516  517  535  536  538  539
  540  541  542  543  544  546  558  559  560  564  565  566  567  572
  576  577  599  601  607  609  628  648  649  650  651  652  653  654
  660  664  665  666  668  669  670  673  674  675  676  677  678  679
  680  681  682  683  684  685  686  687  688  689  690  691  692  693
  694  695  696  697  699  701  702  703  705  706  707  708  709  711
  712  713  714  715  716  717  718  720  722  723  724  727  728  729
  733  734  735  736  738  741  742  743  745  746  747  749  750  751
  752  763  778  779  786  794  795  79

Traditional feature selection complete. Files saved to TeamsExports/Traditional/


C:\Python313\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [  33   34   43   54   55   56   66   70   71   81   83   88   89   96
   97   98   99  100  101  107  116  132  136  137  138  139  141  142
  143  144  145  146  155  158  179  182  183  184  200  214  246  263
  264  265  300  319  320  326  331  333  334  349  350  351  352  354
  358  370  372  384  410  421  422  423  424  425  426  434  453  467
  511  512  513  531  532  534  535  536  537  538  539  540  542  554
  555  556  560  561  562  563  568  572  573  595  597  603  605  624
  643  644  645  646  647  648  649  650  656  660  661  662  664  665
  666  667  669  670  671  672  673  674  675  676  677  678  679  680
  681  682  683  684  685  686  687  688  689  690  691  692  693  695
  697  698  699  701  702  703  704  705  706  707  708  709  710  711
  712  713  714  716  718  719  720  723  724  725  729  730  731  732
  734  737  738  739  741  742  743  74